# Liveness e readiness com queda e recuperação de dependências

José Francisco Alves dos Santos · RMC AI Labs

Objetivo: verificar por HTTP que a API pode continuar respondendo à liveness enquanto a readiness e a operação falham. Interromperemos servidores HTTP substitutos de PostgreSQL e Redis; eles são **dependências simuladas**, não bancos reais. Também reiniciaremos a API com uma dependência parada e mediremos o resultado.

O experimento usa somente serviços locais criados por ele. Não acessa o CRM, não usa credenciais e não requer cloud. O contrato HTTP é executável; os efeitos de Kubernetes descritos no carrossel permanecem documentados, sem validação por este notebook.


## Preparar o ambiente

WSL2 Ubuntu, VS Code conectado ao WSL e Conda. Na raiz do projeto extraído, execute um comando por linha:

```bash
conda create -n devops-probes python=3.12 -y
conda activate devops-probes
python -m pip install .
python -m ipykernel install --user --name devops-probes --display-name "Python 3.12 (devops-probes)"
python --version
python -c "import rmc_probes_lab; print('Importação validada')"
```

No seletor de kernel do VS Code, escolha Python 3.12 (devops-probes). Dependências somente em pyproject.toml. As saídas gravadas abaixo são uma execução de referência, capturada pela execução sequencial das células Python, sem kernel Jupyter neste ambiente. Execute todas as células para produzir sua própria validação em WSL.


In [1]:
from pathlib import Path
import sys, platform, json, csv, hashlib, html, inspect
from datetime import datetime, timezone
from rmc_probes_lab.lab import ProbeLab, experiment

assert sys.version_info[:2] == (3, 12), "Selecione o kernel com Python 3.12."
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Abra o notebook a partir deste projeto."
OUT = ROOT / "artifacts"
OUT.mkdir(exist_ok=True)
print("Python:", platform.python_version())
print("Tipo: simulação local com HTTP real e dependências substitutas")


Python: 3.12.14
Tipo: simulação local com HTTP real e dependências substitutas


## Como o código funciona

`LocalServer` inicia um servidor HTTP em uma porta livre de loopback. `ProbeLab.stop` encerra a dependência e fecha a porta. A API guarda a URL anterior, de modo que sua próxima checagem encontra uma falha de comunicação.

`/live` verifica que a própria API responde. `/ready` consulta as duas dependências e retorna 503 se uma falhar. `/work` representa uma operação que exige ambas. Não inferimos prontidão a partir de liveness.

Cada instância da API possui um UUID e uma geração. As verificações exigem que uma queda ou recuperação de dependência não mude a instância. Só o reinício manual da API deve mudar o UUID. O trecho abaixo é o código usado no experimento, sem simplificação editorial.


In [2]:
print(inspect.getsource(experiment))


def experiment():
    with ProbeLab() as lab:
        initial = lab.measure("01 Saudável", 200)
        lab.stop("postgresql_simulado")
        failed = lab.measure("02 PostgreSQL simulado parado", 503, "postgresql_simulado")
        assert initial["instancia"] == failed["instancia"], "A queda não deve reiniciar a API."
        lab.restart_api()
        restarted = lab.measure("03 API reiniciada com dependência parada",503,"postgresql_simulado")
        assert restarted["instancia"] != failed["instancia"]
        lab.restore("postgresql_simulado")
        restored = lab.measure("04 PostgreSQL simulado recuperado",200)
        assert restored["instancia"] == restarted["instancia"]
        lab.stop("redis_simulado")
        redis_failed = lab.measure("05 Redis simulado parado",503,"redis_simulado")
        assert redis_failed["instancia"] == restored["instancia"]
        lab.restore("redis_simulado")
        final = lab.measure("06 Todas as dependências recuperadas",200)
        assert f

## Executar as seis etapas

1. Medir com todas as dependências saudáveis.
2. Encerrar PostgreSQL simulado e medir.
3. Reiniciar somente a API e medir com a dependência ainda parada.
4. Recuperar PostgreSQL simulado e medir.
5. Encerrar Redis simulado e medir.
6. Recuperar Redis simulado e medir.

As medições são chamadas HTTP de verdade. Os asserts conferem os códigos, a identidade da API e qual dependência falhou. O gerenciador de contexto fecha todos os servidores ao terminar, inclusive se uma verificação falhar.


In [3]:
rows = experiment()
assert len(rows) == 6
assert [r["readiness"] for r in rows] == [200, 503, 503, 200, 503, 200]
assert [r["liveness"] for r in rows] == [200] * 6
assert rows[0]["instancia"] == rows[1]["instancia"]
assert rows[1]["instancia"] != rows[2]["instancia"]
assert len({r["instancia"] for r in rows[2:]}) == 1
print("Conferências concluídas; servidores locais encerrados.")


01 Saudável: live=200, ready=200, work=200, API ativa=True, geração=1
02 PostgreSQL simulado parado: live=200, ready=503, work=503, API ativa=True, geração=1
03 API reiniciada com dependência parada: live=200, ready=503, work=503, API ativa=True, geração=2
04 PostgreSQL simulado recuperado: live=200, ready=200, work=200, API ativa=True, geração=2
05 Redis simulado parado: live=200, ready=503, work=503, API ativa=True, geração=2
06 Todas as dependências recuperadas: live=200, ready=200, work=200, API ativa=True, geração=2
Conferências concluídas; servidores locais encerrados.


## Guardar evidências e rastrear a execução

Os códigos observados são salvos em CSV. O JSON guarda o tipo da demonstração, ambiente, horário, UUIDs e hashes SHA256 do código, do pyproject.toml e do notebook sem saídas. A versão de leitura HTML mostra as seis etapas em uma tabela. Os artefatos não contêm URLs dinâmicas, caminhos do interpretador ou credenciais.


In [4]:
nb_path = ROOT / "notebooks" / "Laboratorio_Liveness_Readiness_Quedas.ipynb"
nb_source = json.loads(nb_path.read_text())
canonical = [{"cell_type":c["cell_type"], "source":"".join(c["source"])} for c in nb_source["cells"]]
digest = lambda data: hashlib.sha256(data).hexdigest()
evidence = {
    "tipo": "simulação local HTTP; dependências substitutas de PostgreSQL e Redis",
    "python": platform.python_version(), "utc": datetime.now(timezone.utc).isoformat(),
    "codigo_sha256": digest((ROOT / "src/rmc_probes_lab/lab.py").read_bytes()),
    "pyproject_sha256": digest((ROOT / "pyproject.toml").read_bytes()),
    "notebook_fonte_sha256": digest(json.dumps(canonical,ensure_ascii=False,sort_keys=True).encode()),
    "resultados": rows,
    "limite": "Não executa bancos reais, Kubernetes, kubelet ou Service.",
}
(OUT / "evidencias.json").write_text(json.dumps(evidence,ensure_ascii=False,indent=2))
fields = ["cenario","liveness","readiness","operacao","api_ativa","geracao_api","instancia","utc"]
with (OUT / "resultados.csv").open("w",newline="") as f:
    writer=csv.DictWriter(f,fieldnames=fields,extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)
table = "".join("<tr>"+"".join("<td>"+html.escape(str(r[k]))+"</td>" for k in fields[:6])+"</tr>" for r in rows)
summary = '<!doctype html><html lang="pt-BR"><meta charset="utf-8"><title>Liveness e readiness</title><style>body{font:18px/1.5 sans-serif;max-width:1100px;margin:40px auto;padding:20px}table{border-collapse:collapse;width:100%}td,th{border:1px solid #ddd;padding:12px;text-align:left}th{background:#e8f3f3}</style><h1>Queda e recuperação de dependências simuladas</h1><p>José Francisco Alves dos Santos · RMC AI Labs. Execução HTTP local com serviços substitutos; não executa bancos reais nem Kubernetes.</p><table><tr>'+''.join('<th>'+html.escape(k)+'</th>' for k in fields[:6])+'</tr>'+table+'</table><p>Liveness permanece 200. Readiness e operação falham com a dependência parada, mesmo após reiniciar a API, e se recuperam após restaurar a dependência.</p></html>'
(OUT / "resumo.html").write_text(summary)
print("Evidências: artifacts/evidencias.json, resultados.csv e resumo.html")


Evidências: artifacts/evidencias.json, resultados.csv e resumo.html


## Conclusão e correspondência com o carrossel

**Demonstrado nesta simulação:** API ativa e liveness200 durante falhas; readiness503 e operação503; reinício da API insuficiente com dependência parada; recuperação200 após restauração; identidade da API preservada durante as quedas e recuperações.

**Documentado:** a readiness controla a prontidão para o tráfego normal de Service e a liveness aciona a política de reinício após os limites configurados. **Pendente por execução:** verificar Ready, EndpointSlices e reinício pelo kubelet em um cluster.

Slides1 e3 encontram evidência local do contrato. Slides2 e5 combinam contrato observado com documentação. Os comandos de kubectl do slide4 são orientações e não foram executados por este notebook.

Fontes: [Probes do Kubernetes](https://kubernetes.io/docs/concepts/workloads/pods/probes/), [http.server Python3.12](https://docs.python.org/3.12/library/http.server.html). A lógica está em `src/rmc_probes_lab/lab.py`.
